# 1.TrH2_processing

1.TrH2 processing。

- 当前文件：`support/Data/01.RawData/PublicData/2023Placozoa/TrH2/1.TrH2_processing.ipynb`
- 原始来源：`Data/01.RawData/PublicData/2023Placozoa/TrH2/1.TrH2_processing.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`BiocManager`, `Matrix`, `Seurat`, `metacell`, `rhdf5`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


## 安装配置环境

## Counts_TrH2

In [ ]:
# 读取Counts原始数据
counts_TrH2 <- readRDS("counts_TrH2.RDS")
counts_TrH2

In [ ]:
dim(counts_TrH2)

In [ ]:
row_names <- rownames(counts_TrH2)
head(row_names)

In [ ]:
col_names <- colnames(counts_TrH2)
head(col_names)

## metacell_fc_TrH2

In [ ]:
metacell_fc_TrH2 <- readRDS("metacell_fc_TrH2.RDS")
metacell_fc_TrH2

In [ ]:
# 12283 genes, 189 metacells
dim(metacell_fc_TrH2)

## metacell_annotation_TrH2

In [ ]:
# 读取 TSV 文件
metacell_annotation_path <- "metacell_annotation_TrH2.tsv"
metacell_annotation_TrH2 <- read.delim(metacell_annotation_path, header = TRUE, sep = "\t")

head(metacell_annotation_TrH2)

In [ ]:
dim(metacell_annotation_TrH2)

In [ ]:
# unique(metacell_annotation_TrH2$cell_type)
# table(metacell_annotation_TrH2$cell_type)

In [ ]:
# unique(metacell_annotation_TrH2$broad_cell_type)
# table(metacell_annotation_TrH2$broad_cell_type)

## Metacell2Allcell

In [ ]:
library(metacell)

In [ ]:
# from the `results_scatlas/` folder in this repository
# load metacell library
library("metacell")
# initialise metacell database using its relative path:
# metacell::scdb_init("data/scdb/", force_reinit = TRUE)
metacell::scdb_init("/share/home/zhangze/zz/NeuralOrigin/Data/01.RawData/PublicData/2023Placozoa/placozoa-cell-type-evolution-code/results_scatlas/data/scdb", force_reinit = TRUE)

In [ ]:
# read the mat object using its ID to refer to it (in this case, the `scdr_TrH2_it2` bit); there ara analogous files for metacells, etc
mat = metacell::scdb_mat("scdr_TrH2_it2")
# sparse matrix available in the mat@mat slot, in this case it contains 16386 genes x 13236 cells
dim(mat@mat)

In [ ]:
mat

In [ ]:
# likewise, you may want to load a mc object containing cell-to-metacell assignments
mc = metacell::scdb_mc("scdr_TrH2_it4")
# the mc@mc slot is a vector with all cells and their associated metacell
length(mc@mc)
# notice that this vector contains only 13151 cells: not all cells in the mat@mat matrix are classified into a metacell

In [ ]:
mc

In [ ]:
# 查看对象的结构
str(mc)

In [ ]:
# 查看对象的摘要，发现变量 mc 是一个 S4 对象
summary(mc)

In [ ]:
# 如果 mc 是一个 S4 对象，可以使用 slotNames() 函数列出对象中的所有槽。
slotNames(mc)

In [ ]:
# 提取特定槽的内容：使用 @ 运算符提取特定槽的内容
head(mc@mc)

In [ ]:
# if you want to know which cell type is annotated to each metacell, check the mc annotation file
# make sure that you load the same version of the metacell clustering as in the mc object, in this case, it4
ctt = read.table("/share/home/zhangze/zz/NeuralOrigin/Data/01.RawData/PublicData/2023Placozoa/placozoa-cell-type-evolution-code/results_scatlas/results_metacell_it4/annotation_mc.TrH2.it4.reordered.tsv", header = TRUE)
head(ctt)
# metacell	cell_type	color	broad_cell_type	metacell_it2
# 1	lipophil	khaki2	lipophil	1
# 2	lipophil	khaki2	lipophil	2
# 3	lipophil	khaki2	lipophil	3
# 4	lipophil	khaki2	lipophil	4
# 5	lipophil	khaki2	lipophil	5

In [ ]:
# 提取 mc 槽中的元细胞编号和 cell_names 槽中的细胞名称
cell_to_metacell <- data.frame(
  cell_name = names(mc@mc),  # 细胞名称
  metacell_id = mc@mc        # 对应的元细胞编号
)

# 查看前几行结果
head(cell_to_metacell)

In [ ]:
dim(cell_to_metacell)

In [ ]:
# 如果需要，将列名统一为 metacell_id
colnames(metacell_annotation_TrH2)[which(colnames(metacell_annotation_TrH2) == "metacell_it2")] <- "metacell_id"

In [ ]:
head(cell_to_metacell)

In [ ]:
head(metacell_annotation_TrH2)

In [ ]:
# 合并两个数据框
merged_data <- merge(cell_to_metacell, metacell_annotation_TrH2, by = "metacell_id", all.x = TRUE)

# 查看合并后的数据框
head(merged_data)
dim(merged_data)


In [ ]:
# 提取 merged_data 中的细胞名称
selected_cells <- merged_data$cell_name
head(selected_cells)

In [ ]:
# 过滤 counts_TrH2 中的列，只保留 selected_cells 中的细胞
counts_TrH2_filtered <- counts_TrH2[, selected_cells]
head(counts_TrH2_filtered)

In [ ]:
# 查看过滤后的矩阵的维度
dim(counts_TrH2_filtered)

In [ ]:
col_names_filtered <- colnames(counts_TrH2_filtered)
head(col_names_filtered)

In [ ]:
# 创建一个列表，包含 counts_TrH2_filtered 和 merged_data
combined_data <- list(
  counts = counts_TrH2_filtered,
  metadata = merged_data
)

In [ ]:
# 保存为 .rds 文件
saveRDS(combined_data, file = "sc_TrH2.rds")

## Save to Seurat

In [ ]:
# 读取保存的 .rds 文件
loaded_data <- readRDS("sc_TrH2.rds")
loaded_data

# 访问 counts_TrH2_filtered 矩阵
counts_TrH2_filtered_loaded <- loaded_data$counts

# 访问 merged_data 数据框
merged_data_loaded <- loaded_data$metadata

# 查看加载后的数据
print(dim(counts_TrH2_filtered_loaded))  # 检查 counts 矩阵的维度
print(head(merged_data_loaded))  # 查看 metadata 的前几行

In [ ]:
library(Seurat)

# 创建 Seurat 对象
seurat_object <- CreateSeuratObject(counts = counts_TrH2_filtered_loaded)

# 将 metadata 添加到 Seurat 对象中
seurat_object <- AddMetaData(seurat_object, metadata = merged_data_loaded)

# 保存 Seurat 对象
saveRDS(seurat_object, file = "sc_seurat_TrH2.rds")

## RDS to h5ad

In [ ]:
# if (!requireNamespace("BiocManager", quietly = TRUE))
#   install.packages("BiocManager")
# BiocManager::install("SeuratObject")
# BiocManager::install("Seurat")

library(Seurat)

# R code for converting RDS to h5
library(rhdf5)
library(Matrix)

# 保存为 h5 矩阵
save_h5mat = function(mat, fp_h5, feature_type, genome=""){
  # save sparse.mat ('dgCMatrix' format) into a h5 file
  # ======= Test code ======
  # tmp = Seurat::Read10X_h5(fp_h5)
  # all(tmp@x == mat@x)
  # all(tmp@i == mat@i)
  # all(tmp@p == mat@p)
  
  message(fp_h5)
  
  h5createFile(fp_h5)
  root = "matrix"
  h5createGroup(fp_h5, root)
  
  h5write(dim(mat), fp_h5, paste(root, "shape", sep='/'))
  h5write(mat@x, fp_h5, paste(root, "data", sep='/'))
  h5write(mat@i, fp_h5, paste(root, "indices", sep='/'))  # mat@i - 1 ?
  h5write(mat@p, fp_h5, paste(root, "indptr", sep='/'))
  h5write(colnames(mat), fp_h5, paste(root, "barcodes", sep='/'))
  
  
  feat_root = paste(root, "features", sep='/')
  h5createGroup(fp_h5, feat_root)
  
  h5write(rownames(mat), fp_h5, paste(feat_root, "id", sep='/'))
  h5write(rownames(mat), fp_h5, paste(feat_root, "name", sep='/'))
  
  h5write(rep(feature_type, dim(mat)[1]),
          fp_h5, paste(feat_root, "feature_type", sep='/'))
  
  h5write(rep("", dim(mat)[1]),
          fp_h5, paste(feat_root, "derivation", sep='/'))
  h5write(rep(genome, dim(mat)[1]),  # "mm10"
          fp_h5, paste(feat_root, "genome", sep='/'))
  h5write(c("genome", "derivation"),
          fp_h5, paste(feat_root, "_all_tag_keys", sep='/'))
  
  h5closeAll()
  message("Done!")
}

# save_h5mat_peak = function(mat, fp_h5, genome=""){
#   save_h5mat(mat, fp_h5, feature_type = "Peaks", genome = genome)
# }

save_h5mat_gex = function(mat, fp_h5, genome=""){
  save_h5mat(mat, fp_h5, feature_type = "Gene Expression", genome = genome)
}

In [ ]:
## save the raw-counts in a Seurat-object "seurat_obj"
# 加载 RDS 数据
SCT_UMI_expression_matrix <- readRDS("sc_seurat_TrH2.rds")
seurat_object <- SCT_UMI_expression_matrix
seurat_object

In [ ]:
# 查看Seurat对象的基本信息
print(seurat_object)

In [ ]:
# 查看包含的 assays
Assays(seurat_object)

In [ ]:
# 查看元数据
head(seurat_object@meta.data)

In [ ]:
# 定义输出文件路径模板
h5_path <- "sc_TrH2.matrix.raw.h5"
csv_path <- "sc_TrH2.metadata.csv"

# save the raw-counts in a Seurat-object "seurat_obj"

# 提取计数矩阵, 对于 Assay5 类型，你需要使用 GetAssayData() 函数来提取原始计数矩阵：
mat <- GetAssayData(seurat_object, slot = "counts")
# mat = seurat_obj[["RNA"]]@counts
save_h5mat_gex(mat, h5_path, genome="")

# save the meta-data into a csv file:
meta_data = seurat_object@meta.data
write.csv(meta_data, csv_path)